# 📊 CreditIQ — Exploratory Data Analysis

**Goal:** Understand the dataset before building any models.
- What does the data look like?
- How much is missing?
- How imbalanced is the target?
- Which features correlate with default?

> EDA is not optional — it determines every preprocessing decision you make.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make plots look clean
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')

print('Libraries loaded ✅')

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/cs-training.csv', index_col=0)
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Basic statistics for each column
df.describe().round(2)

## 2. Missing Values

Missing data is one of the first things to check. Real-world datasets are always messy.

In [ ]:
# Count and percentage of missing values per column
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})
missing = missing[missing['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print(missing)

# Observations:
# MonthlyIncome is missing for ~20% of rows
# NumberOfDependents is missing for ~2.5% of rows
# Strategy: median for income (skewed), mode for dependents (discrete)

In [ ]:
# Visualise missing values
fig, ax = plt.subplots(figsize=(8, 4))
missing['Missing %'].plot(kind='barh', ax=ax, color='#e74c3c')
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Values by Column')
for i, v in enumerate(missing['Missing %']):
    ax.text(v + 0.2, i, f'{v}%', va='center')
plt.tight_layout()
plt.show()

## 3. Target Variable — Class Imbalance

The most important thing to check for classification problems.

In [ ]:
target_counts = df['SeriousDlqin2yrs'].value_counts()
target_pct    = df['SeriousDlqin2yrs'].value_counts(normalize=True).round(3) * 100

print('Target Distribution:')
print(pd.DataFrame({'Count': target_counts, 'Percentage': target_pct}))

imbalance_ratio = target_counts[0] / target_counts[1]
print(f'\nImbalance ratio: {imbalance_ratio:.1f}:1 (non-default:default)')
print('\n⚠️  This is why we need SMOTE — a naive model would just predict 0 always!')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
target_counts.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[0].set_title('Loan Default Distribution')
axes[0].set_xticklabels(['No Default (0)', 'Default (1)'], rotation=0)
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(
    target_counts,
    labels=['No Default', 'Default'],
    colors=['#2ecc71', '#e74c3c'],
    autopct='%1.1f%%',
    startangle=90
)
axes[1].set_title('Class Proportion')

plt.tight_layout()
plt.show()

## 4. Feature Distributions

Understanding each feature's distribution helps with preprocessing decisions.

In [ ]:
# Drop target and plot all features
features = df.drop(columns=['SeriousDlqin2yrs'])

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(features.columns):
    axes[i].hist(df[col].dropna(), bins=50, color='#3498db', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_ylabel('Count')

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Notice: MonthlyIncome and DebtRatio are heavily right-skewed
# Notice: Age is roughly normal (good!)

## 5. Feature vs. Target — Who's More Likely to Default?

In [ ]:
# Age distribution by default status
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age
df.groupby('SeriousDlqin2yrs')['age'].plot(
    kind='hist', bins=40, alpha=0.6, ax=axes[0]
)
axes[0].set_title('Age Distribution by Default Status')
axes[0].legend(['No Default', 'Default'])
axes[0].set_xlabel('Age')

# Monthly Income
df_income = df[df['MonthlyIncome'] < df['MonthlyIncome'].quantile(0.95)]  # remove extreme outliers for viz
df_income.groupby('SeriousDlqin2yrs')['MonthlyIncome'].plot(
    kind='hist', bins=40, alpha=0.6, ax=axes[1]
)
axes[1].set_title('Monthly Income by Default Status')
axes[1].legend(['No Default', 'Default'])
axes[1].set_xlabel('Monthly Income')

plt.tight_layout()
plt.show()

In [ ]:
# Default rate by age group
df['AgeGroup'] = pd.cut(df['age'], bins=[0, 25, 35, 50, 65, 120],
                         labels=['<25', '25-35', '35-50', '50-65', '65+'])

default_by_age = df.groupby('AgeGroup', observed=True)['SeriousDlqin2yrs'].mean() * 100

fig, ax = plt.subplots(figsize=(8, 4))
default_by_age.plot(kind='bar', ax=ax, color='#e74c3c', edgecolor='white')
ax.set_title('Default Rate by Age Group')
ax.set_ylabel('Default Rate (%)')
ax.set_xlabel('Age Group')
for i, v in enumerate(default_by_age):
    ax.text(i, v + 0.1, f'{v:.1f}%', ha='center', fontweight='bold')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Young borrowers default more — this will be a useful feature!

## 6. Correlation Analysis

In [ ]:
# Correlation with target variable
corr_with_target = df.corr(numeric_only=True)['SeriousDlqin2yrs'].drop('SeriousDlqin2yrs').sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in corr_with_target]
corr_with_target.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Feature Correlation with Default (Target)', fontweight='bold')
ax.set_xlabel('Pearson Correlation')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

# Red = positively correlated with default (more past-dues = higher default risk)
# Green = negatively correlated (older age = lower default risk)

In [ ]:
# Full correlation heatmap
plt.figure(figsize=(12, 8))
corr_matrix = df.drop(columns=['AgeGroup']).corr(numeric_only=True)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # hide upper triangle (redundant)

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Outlier Detection

In [ ]:
# Box plots to spot outliers
cols_to_check = ['RevolvingUtilizationOfUnsecuredLines', 'DebtRatio', 'MonthlyIncome', 'age']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for i, col in enumerate(cols_to_check):
    df[col].plot(kind='box', ax=axes[i], color='#3498db')
    axes[i].set_title(col, fontsize=9)

plt.suptitle('Outlier Detection via Box Plots', fontweight='bold')
plt.tight_layout()
plt.show()

# RevolvingUtilization has values WAY above 1.0 (should be 0-1 ratio)
# DebtRatio has extreme values — these will be capped in preprocessing
print('\nExtreme values:')
print(f'Max RevolvingUtilization: {df["RevolvingUtilizationOfUnsecuredLines"].max():.2f}')
print(f'Max DebtRatio: {df["DebtRatio"].max():.2f}')
print(f'Max MonthlyIncome: {df["MonthlyIncome"].max():.2f}')

## 8. EDA Summary & Preprocessing Plan

Based on this analysis, here's what we'll do in the modeling notebook:

| Issue Found | Preprocessing Decision |
|-------------|------------------------|
| MonthlyIncome missing ~20% | Fill with **median** (skewed dist.) |
| NumberOfDependents missing ~2.5% | Fill with **mode** |
| DebtRatio & RevolvingUtilization extreme outliers | **Cap at 99th percentile** |
| Target is 93.5% / 6.5% imbalanced | Apply **SMOTE** oversampling |
| Past-due columns all correlated | Engineer **TotalPastDue** combined feature |
| Young borrowers default more | Engineer **AgeGroup** feature |

→ Now open `02_modeling.ipynb` to build the models.